# GRPO Notebook 2 — Full Training (200 problems)

**What this does:**
Trains Qwen2-1.5B-Instruct (INT8 + LoRA) on 200 GSM8K training problems
for 1 epoch using GRPO with binary correctness reward.
Saves the fine-tuned LoRA adapter to Google Drive.

**Model:** Qwen/Qwen2-1.5B-Instruct | **Quant:** INT8 | 

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────

!pip install -q "transformers>=4.40.0" "trl>=0.8.6" "peft>=0.10.0" "bitsandbytes>=0.43.0" "accelerate>=0.29.0" "datasets>=2.19.0"
print('Dependencies installed')

In [ ]:
# ── CELL 2: GPU check ─────────────────────────────────────────────────────────

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── CELL 3: Mount Drive ───────────────────────────────────────────────────────
# Checkpoint saves here after every 50 steps so disconnection doesn't lose training

from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR  = '/content/drive/MyDrive/quantization_research/grpo'
CKPT_DIR    = os.path.join(OUTPUT_DIR, 'full_ckpt')
LOG_PATH    = os.path.join(OUTPUT_DIR, 'training_log.csv')
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')
print(f'Training log:   {LOG_PATH}')

In [ ]:
# ── CELL 4: Imports ───────────────────────────────────────────────────────────

import re, gc, random, time, os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer
print('Imports OK')

In [ ]:
# ── CELL 5: Config ────────────────────────────────────────────────────────────
# TRAIN_N = 200 is the full training run
# Everything else matches smoke test exactly

MODEL_ID   = 'Qwen/Qwen2-1.5B-Instruct'
SEED       = 42
TRAIN_N    = 200    # full training — 200 problems from GSM8K train split
MAX_TOKENS = 1024
GROUP_SIZE = 4      # G responses per prompt
EPOCHS     = 1
LR         = 1e-5

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Full training: {TRAIN_N} problems, G={GROUP_SIZE}, {EPOCHS} epoch')

In [ ]:
# ── CELL 6: Load GSM8K train split and sample 200 problems ────────────────────

print('Loading GSM8K train split...')
ds_train = load_dataset('openai/gsm8k', 'main', split='train')
print(f'Total available: {len(ds_train)}')

random.seed(SEED)
train_indices  = random.sample(range(len(ds_train)), TRAIN_N)
train_problems = [ds_train[i] for i in train_indices]

print(f'Sampled {len(train_problems)} training problems')

In [ ]:
# ── CELL 7: Answer extraction + prompt formatting ─────────────────────────────

def extract_ground_truth(answer_str):
    """Extract number after #### from GSM8K gold answer."""
    match = re.search(r'####\s*([\d,\.\-]+)', answer_str)
    if match:
        return match.group(1).replace(',', '').strip()
    return None

def extract_prediction(output_str):
    """Extract final numeric answer from model output."""
    m = re.search(r'\\boxed\{([\d,\.\-]+)\}', output_str)
    if m: return m.group(1).replace(',', '').strip()
    m = re.search(r'####\s*([\d,\.\-]+)', output_str)
    if m: return m.group(1).replace(',', '').strip()
    nums = re.findall(r'\b([\d,\.]+)\b', output_str)
    if nums: return nums[-1].replace(',', '').strip()
    return None

def format_prompt(question):
    """Format as Qwen2-Instruct chat prompt — matches Shaswat's format."""
    return (
        '<|im_start|>system\n'
        'You are a math problem solver. Show your reasoning step by step. '
        'End your answer with the final number in \\boxed{}.\n'
        '<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )

def build_grpo_dataset(problems):
    rows = []
    for p in problems:
        gt = extract_ground_truth(p['answer'])
        if gt is None: continue
        rows.append({'prompt': format_prompt(p['question']), 'ground_truth': gt})
    return Dataset.from_list(rows)

train_dataset = build_grpo_dataset(train_problems)
print(f'Training dataset: {len(train_dataset)} problems')

In [ ]:
# ── CELL 8: Reward function with step logging ──────────────────────────────────

reward_log = []  # collects (step, n_correct, n_total) for saving
step_counter = [0]

def reward_fn(prompts, completions, ground_truth=None, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        gt   = ground_truth[i % len(ground_truth)] if ground_truth else None
        pred = extract_prediction(completion)
        correct = (pred is not None and gt is not None
                   and pred.strip() == gt.strip())
        rewards.append(1.0 if correct else 0.0)

    n_correct = int(sum(rewards))
    step_counter[0] += 1
    reward_log.append({'step': step_counter[0],
                       'n_correct': n_correct,
                       'n_total': len(rewards),
                       'reward_pct': 100*n_correct/max(len(rewards),1)})

    # Save log to Drive every 10 steps so progress is never lost
    if step_counter[0] % 10 == 0:
        pd.DataFrame(reward_log).to_csv(LOG_PATH, index=False)
        print(f'  [step {step_counter[0]}] reward: {n_correct}/{len(rewards)} '
              f'({100*n_correct/max(len(rewards),1):.0f}%) — log saved')
    else:
        print(f'  [step {step_counter[0]}] reward: {n_correct}/{len(rewards)} '
              f'({100*n_correct/max(len(rewards),1):.0f}%)')

    return rewards

print('Reward function ready')

In [ ]:
# ── CELL 9: Load model with INT8 quantization ─────────────────────────────────

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

print('Loading model in INT8...')
bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
)
print(f'GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Prepare for kbit training (enables gradient flow through quantized layers)
model = prepare_model_for_kbit_training(model)
print('Model ready')

In [ ]:
# ── CELL 10: GRPO training config ─────────────────────────────────────────────

grpo_cfg = GRPOConfig(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations             = GROUP_SIZE,
    max_completion_length       = MAX_TOKENS,
    temperature                 = 0.7,
    learning_rate               = LR,
    optim                       = 'adamw_torch',
    gradient_checkpointing      = True,
    fp16                        = False,   # causes conflict with INT8 + BFloat16
    bf16                        = True,    # correct for Qwen2
    logging_steps               = 5,
    save_steps                  = 50,      # checkpoint every 50 steps to /content/
    save_total_limit            = 2,
    report_to                   = 'none',
)

# LoRA config — passed to trainer directly in TRL 1.2.0
lora_cfg = LoraConfig(
    r              = 8,
    lora_alpha     = 32,
    target_modules = ['q_proj', 'v_proj'],
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
)

print('Configs ready')

In [ ]:
# ── CELL 11: Run full training ─────────────────────────────────────────────────
print('='*55)
print(f'GRPO FULL TRAINING — {TRAIN_N} problems, {EPOCHS} epoch')
print('='*55)
t0 = time.time()

trainer = GRPOTrainer(
    model            = model,
    args             = grpo_cfg,
    train_dataset    = train_dataset,
    reward_funcs     = reward_fn,
    processing_class = tokenizer,   # renamed from 'tokenizer' in TRL 1.2.0
    peft_config      = lora_cfg,    # LoRA applied internally by trainer
)
trainer.train()

elapsed  = (time.time() - t0) / 60
peak_mem = torch.cuda.max_memory_allocated() / 1e9

print('='*55)
print(f'TRAINING COMPLETE')
print(f'  Time:      {elapsed:.1f} min')
print(f'  Peak VRAM: {peak_mem:.2f} GB')
print('='*55)

In [ ]:
# ── CELL 12: Save final model to Drive ────────────────────────────────────────

ADAPTER_PATH = os.path.join(OUTPUT_DIR, 'final_adapter')
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved to: {ADAPTER_PATH}')

# Also save the final reward log
pd.DataFrame(reward_log).to_csv(LOG_PATH, index=False)
print(f'Training log saved to: {LOG_PATH}')
print(f'Log has {len(reward_log)} entries')

In [ ]:
# ── CELL 13: Training summary ──────────────────────────────────────────────────

import pandas as pd
log_df = pd.DataFrame(reward_log)

print('Training reward summary:')
print(f'  First 10 steps mean reward: {log_df.head(10)["reward_pct"].mean():.1f}%')
print(f'  Last 10 steps mean reward:  {log_df.tail(10)["reward_pct"].mean():.1f}%')
print(f'  Overall mean reward:        {log_df["reward_pct"].mean():.1f}%')
print()
print('Full log:')
print(log_df.to_string())

In [ ]:
# ── CELL 14: Cleanup ──────────────────────────────────────────────────────────

del trainer, model
gc.collect()
torch.cuda.empty_cache()
print(f'GPU mem after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Training done. Open Notebook 3 for evaluation.')

In [ ]:
#Evaluation

# GRPO Notebook 3 — Evaluation


**What this does:**
Loads the GRPO fine-tuned adapter, runs it on the exact same 30 test problems
Shaswat used (greedy decoding), and records accuracy by difficulty.
That accuracy is the training-time upper bound for the paper.

**The number to report:** Overall accuracy — compare this against best
inference-time result (SC N=5 at 46.7%) to get the GAP.

**Model:** Qwen/Qwen2-1.5B-Instruct + GRPO LoRA | **Quant:** INT8 |

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────

!pip install -q "transformers>=4.40.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" "accelerate>=0.29.0" "datasets>=2.19.0"
print('Dependencies installed')

In [ ]:
# ── CELL 2: GPU check ─────────────────────────────────────────────────────────

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── CELL 3: Mount Drive ───────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR   = '/content/drive/MyDrive/quantization_research/grpo'
ADAPTER_PATH = os.path.join(OUTPUT_DIR, 'final_adapter')
EVAL_PATH    = os.path.join(OUTPUT_DIR, 'grpo_eval_results.csv')

# Confirm adapter exists before proceeding
if not os.path.exists(ADAPTER_PATH):
    raise FileNotFoundError(
        f'Adapter not found at {ADAPTER_PATH}. '
        'Run Notebook 2 first and confirm it saved successfully.'
    )
print(f'Adapter found: {ADAPTER_PATH}')
print(f'Results will save to: {EVAL_PATH}')

In [ ]:
# ── CELL 4: Imports ───────────────────────────────────────────────────────────

import re, gc, time
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel  # PeftModel still works fine for loading saved adapter
print('Imports OK')

In [ ]:
# ── CELL 5: Test problem IDs — Shaswat's exact 30 problems ────────────────────
MODEL_ID   = 'Qwen/Qwen2-1.5B-Instruct'
MAX_TOKENS = 512
SEED       = 42

# Shaswat's exact test problem IDs by difficulty
TEST_IDS = {
    'easy':   [473, 624, 549, 1028, 958, 263, 493, 885, 1103, 450,
               591, 235, 29, 105, 199, 227, 326, 465, 722, 728],
    'medium': [1089, 359, 872, 209, 850, 457, 718, 61, 267, 187,
               586, 438, 175, 433, 775, 830, 858, 997, 1035, 1202],
    'hard':   [676, 1314, 361, 721, 629, 814, 1041, 1044, 1235, 1310],
}
ALL_TEST_IDS = TEST_IDS['easy'] + TEST_IDS['medium'] + TEST_IDS['hard']
print(f'Test set: {len(ALL_TEST_IDS)} problems '
      f'({len(TEST_IDS["easy"])} easy, '
      f'{len(TEST_IDS["medium"])} medium, '
      f'{len(TEST_IDS["hard"])} hard)')

In [ ]:
# ── CELL 6: Load test problems from GSM8K test split ──────────────────────────

print('Loading GSM8K test split...')
ds_test = load_dataset('openai/gsm8k', 'main', split='test')
print(f'Total test problems: {len(ds_test)}')

# Filter to Shaswat's problem IDs
id_set = set(ALL_TEST_IDS)
test_problems = []
for idx, row in enumerate(ds_test):
    if idx in id_set:
        # Determine difficulty
        if idx in TEST_IDS['easy']:   diff = 'easy'
        elif idx in TEST_IDS['medium']: diff = 'medium'
        else:                           diff = 'hard'
        test_problems.append({
            'problem_id': idx,
            'difficulty': diff,
            'question':   row['question'],
            'answer':     row['answer'],
        })

print(f'Loaded {len(test_problems)} test problems')
# Sanity check — should match exactly
if len(test_problems) != len(ALL_TEST_IDS):
    print(f'WARNING: Expected {len(ALL_TEST_IDS)}, got {len(test_problems)}')

In [ ]:
# ── CELL 7: Answer extraction + prompt formatting ─────────────────────────────

def extract_ground_truth(answer_str):
    match = re.search(r'####\s*([\d,\.\-]+)', answer_str)
    if match:
        return match.group(1).replace(',', '').strip()
    return None

def extract_prediction(output_str):
    m = re.search(r'\\boxed\{([\d,\.\-]+)\}', output_str)
    if m: return m.group(1).replace(',', '').strip()
    m = re.search(r'####\s*([\d,\.\-]+)', output_str)
    if m: return m.group(1).replace(',', '').strip()
    nums = re.findall(r'\b([\d,\.]+)\b', output_str)
    if nums: return nums[-1].replace(',', '').strip()
    return None

def format_prompt(question):
    return (
        '<|im_start|>system\n'
        'You are a math problem solver. Show your reasoning step by step. '
        'End your answer with the final number in \\boxed{}.\n'
        '<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )

print('Helpers ready')

In [ ]:
# Load the GRPO-trained model directly from saved weights
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

ADAPTER_PATH = '/content/drive/MyDrive/quantization_research/grpo/final_adapter'

print('Loading GRPO model directly from saved weights...')
bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    ADAPTER_PATH,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Model loaded — proceeding to evaluation')

In [ ]:
# ── CELL 9: Run evaluation — greedy decoding on all 30 test problems ──────────

results = []

print('='*55)
print('GRPO EVALUATION — greedy decoding')
print('='*55)
t0 = time.time()

for i, prob in enumerate(test_problems):
    prompt = format_prompt(prob['question'])
    gt     = extract_ground_truth(prob['answer'])

    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = MAX_TOKENS,
            do_sample      = False,   # greedy — deterministic
            pad_token_id   = tokenizer.eos_token_id,
        )

    # Decode only the generated part (not the prompt)
    generated = tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    pred    = extract_prediction(generated)
    correct = (pred is not None and gt is not None
               and pred.strip() == gt.strip())
    elapsed = (time.time() - t0) / 60

    result = {
        'problem_id':   prob['problem_id'],
        'difficulty':   prob['difficulty'],
        'ground_truth': gt,
        'predicted':    pred,
        'correct':      correct,
        'raw_output':   generated[:500],  # first 500 chars for inspection
        'quant_type':   'W8A8_GRPO',
        'decoding':     'greedy',
    }
    results.append(result)

    # Save after every problem
    pd.DataFrame(results).to_csv(EVAL_PATH, index=False)

    print(f'  [{i+1:2d}/{len(test_problems)}] pid={prob["problem_id"]:<5}'
          f'diff={prob["difficulty"]:<7} '
          f'pred={str(pred):<12} gt={str(gt):<10} '
          f'correct={correct}  elapsed={elapsed:.0f}min')

print('='*55)
print('Evaluation complete')

In [ ]:
# ── CELL 10: Results summary ───────────────────────────────────────────────────

df = pd.DataFrame(results)

# Overall accuracy
total   = len(df)
correct = df['correct'].sum()
overall = 100 * correct / total

print('='*55)
print('GRPO RESULTS SUMMARY (W8A8 + LoRA fine-tuned)')
print('='*55)
print(f'Overall:  {overall:.1f}%  ({correct}/{total})')
for diff in ['easy', 'medium', 'hard']:
    sub = df[df['difficulty'] == diff]
    acc = 100 * sub['correct'].sum() / len(sub)
    print(f'  {diff:<8}: {acc:.1f}%  ({sub["correct"].sum()}/{len(sub)})')

print()
print('Comparison (from Shaswat results — 50 problems):')
print(f'  BF16  Greedy:           42.0%  (full precision baseline)')
print(f'  BF16  Self-Consistency: 56.0%  (best overall)')
print(f'  W8A16 Greedy:           46.0%  (quantization — greedy)')
print(f'  W8A16 Self-Consistency: 52.0%  (best inference-time at W8A16)')
print(f'  W4A16 Greedy:           50.0%  (aggressive compression — greedy)')
print(f'  W4A16 Self-Consistency: 38.0%  (SC hurts at W4A16)')
print(f'  W8A16 GRPO (yours):     {overall:.1f}%  <-- YOUR NUMBER')
print()

# Compare against best inference-time at W8A16 which is SC at 52.0%
gap_vs_sc = overall - 52.0
if gap_vs_sc > 0:
    print(f'GAP: GRPO is {gap_vs_sc:.1f}% ABOVE W8A16 SC')
    print('     --> Retraining provides additional benefit beyond inference-time methods')
elif abs(gap_vs_sc) < 2:
    print(f'GAP: GRPO is within noise of W8A16 SC ({gap_vs_sc:.1f}%)')
    print('     --> Inference-time recovery matches retraining at W8A16')
else:
    print(f'GAP: GRPO is {abs(gap_vs_sc):.1f}% BELOW W8A16 SC')
    print('     --> Inference-time SC outperforms retraining on this subset')
print('='*55)
print(f'Results saved to: {EVAL_PATH}')

In [ ]:
# ── CELL 11: Wrong answer inspection ──────────────────────────────────────────

wrong = df[df['correct'] == False][['problem_id', 'difficulty', 'predicted', 'ground_truth']]
print(f'Wrong answers ({len(wrong)}/{total}):')
print(wrong.to_string(index=False))

print()
print('Reference — Shaswat W8A16 results (50 problems):')
print('  Greedy:           46.0%  (27 wrong)')
print('  Self-Consistency: 52.0%  (24 wrong)')
print()
print(f'GRPO wrong count: {len(wrong)}/{total}')
print(f'GRPO wrong IDs:   {sorted(wrong["problem_id"].tolist())}')

In [ ]:
# ── CELL 12: Save final results and cleanup ────────────────────────────────────

# Final save of complete results
df.to_csv(EVAL_PATH, index=False)
print(f'Final results saved: {EVAL_PATH}')
print(f'Rows: {len(df)}')

# Cleanup GPU
del model, base_model
gc.collect()
torch.cuda.empty_cache()
print(f'GPU mem after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print()
print('DONE. Share the overall accuracy number with the team.')
print('This goes into Table 5.6 (GRPO Upper Bound Grid) in the paper.')